# PaReco results analysis

Thin driver notebook. All logic lives in the `pareco_analysis` package; each cell below runs one step.
Settings (paths, variant pairs, the hard date limit) are in `pareco_analysis/config.py`.

The same steps can be run without Jupyter: `python run_analysis.py all`.

In [ ]:
from pareco_analysis import config
from pareco_analysis.audit_export import export_all_pickles
from pareco_analysis.audit_combine import combine_all_audits
from pareco_analysis.data_division import create_division_plan, print_division_report
from pareco_analysis.descriptive_stats import build_descriptive_results
from pareco_analysis.figure6 import generate_figure6
from pareco_analysis.technical_lag import build_technical_lag_data
from pareco_analysis import plots

RESULTS = config.RESULTS_ROOT
print("Results folder:", RESULTS)
print("Pickle folder: ", config.PICKLE_DIR)

## 1. Export the PaReco pickles to JSON and audit CSVs
Creates `Results/<index>_<repo>/` with `classification_audit.csv`, `file_audit.csv` and `hunk_audit.csv`.
Pass `audit_prs=[14635]` to export only specific PRs.

In [ ]:
export_all_pickles(config.PICKLE_DIR, RESULTS)

## 2. Combine the per-folder audits into global audits

In [ ]:
combine_all_audits(RESULTS)

## 3. Attribute every result folder to a variant pair and direction
The direction follows from each folder's source repository (renamed repositories are matched through `config.RENAMED_REPOS`).
The pair comes from the PaReco notebook's saved `interval_plan*.csv` files next to the pickles when present, otherwise from the collection order of `config.VARIANT_PAIRS` within each direction.
Each direction may have its own numbering (as GACPD's `dsep3_` / `dsep3R_` runs do), so folders are identified by name.

In [ ]:
plan = create_division_plan(RESULTS, config.VARIANT_PAIRS)
print_division_report(plan)

## 4. Table 1, Figure 4 and Figure 5
Table 1 / Figure 4 count PRs (classification audit). Figure 5 uses the file audit only.

In [ ]:
stats = build_descriptive_results(plan, RESULTS)
plots.plot_figure4(stats.per_direction_counts, stats.class_order,
                   RESULTS / "figure4_patch_distribution.png", show=True)
plots.plot_figure5(stats.figure5_rows,
                   RESULTS / "figure5_interesting_file_proportions.png", show=True)

## 5. Figure 6 — patch classes per variant pair
`upstream=None` processes every pair (and checks the total against the global audits).
For upstreams that occur more than once use `occurrence=2` or an exact `pair_list_index`.

In [ ]:
figure6_rows = generate_figure6(plan, stats, upstream="spack/spack",
                                results_root=RESULTS, show=True)

## 6. Figures 7 and 8 — technical lag of missed (MO) patches
Needs GitHub access (a `tokens.txt` with one token per line next to this notebook is used if present).
Answers are cached in `Results/TechnicalLag/technical_lag_api_cache.json`.

In [ ]:
# Optional: pin a target HEAD, e.g. {"owner/repo": {"sha": "..."}} or {"owner/repo": {"date": "2026-08-20T00:00:00Z"}}
HEAD_OVERRIDES = {}

lag = build_technical_lag_data(plan, stats, RESULTS, head_overrides=HEAD_OVERRIDES)
plots.plot_figure7(lag["lag_rows"], lag["output_dir"] / "figure7_patch_technical_lag.png", show=True)
plots.plot_figure8(lag["target_rows"], lag["output_dir"] / "figure8_target_variant_mean_technical_lag.png", show=True)